In [ ]:
USR_NAME = 'thebeo2004'
REPO_NAME = 'AMBER'

!git clone https://{USR_NAME}@github.com/{USR_NAME}/{REPO_NAME}.git
%cd /kaggle/working/{REPO_NAME}
!git checkout cascade-anomaly
!git submodule update --init --recursive

In [ ]:
!uv pip install --system -q -r pyproject.toml
!pip install -q ninja
%cd detrex/detectron2
!uv pip install --system -q -e . --no-build-isolation
%cd ..
!uv pip install --system -q -e . --no-build-isolation
%cd /kaggle/working/{REPO_NAME}/anomalib
!uv pip install --system -q -e .

In [ ]:
%%writefile /kaggle/working/run_inference.py

import os
import sys
import numpy as np

os.chdir('/kaggle/working/AMBER')

# 2. Đưa SAGE vào sys.path để đảm bảo import tuyệt đối chính xác
if '/kaggle/working/AMBER' not in sys.path:
    sys.path.insert(0, '/kaggle/working/AMBER')

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
from src.memory_bank.binary_classification import binary_classification_inference_aamb

raddino_checkpoint_path = "/kaggle/input/models/thebeo182004/raddino-vitb14-topad16/pytorch/default/1/rad_dino_vitb14_detrex_ready.pth"

test_normal_dir = "/kaggle/input/datasets/thebeo182004/medanomaly-vindrcxr/medi_anomaly/test/normal"
test_abnormal_dir = "/kaggle/input/datasets/thebeo182004/medanomaly-vindrcxr/medi_anomaly/test/abnormal"

test_normal_imgs = [
    os.path.join(test_normal_dir, img) for img in os.listdir(test_normal_dir)
]

test_abnormal_imgs = [
    os.path.join(test_abnormal_dir, img) for img in os.listdir(test_abnormal_dir)
]

print(f"Total Normal Images: {len(test_normal_imgs)}, Total Abnormal Images: {len(test_abnormal_imgs)}")

memory_bank_dirs = [
    "/kaggle/input/notebooks/thebeo182004/anomalyraddinov2-vindrcxr-50-1-0-1-1",
    "/kaggle/input/notebooks/thebeo182004/anomalyraddinov2-vindrcxr-100-0-5-1-2",
    "/kaggle/input/notebooks/thebeo182004/anomalyraddinov2-vindrcxr-250-0-2-1-3",
    "/kaggle/input/notebooks/thebeo182004/anomalyraddinov2-vindrcxr-500-0-1-1-4",
    "/kaggle/input/notebooks/thebeo182004/anomalyraddinov2-vindrcxr-1000-0-05-1-5",
    "/kaggle/input/notebooks/thebeo2004/anomalyraddinov2-vindrcxr-1500-0-03-1-6",
    "/kaggle/input/notebooks/thebeo182004/anomalyraddinov2-vindrcxr-2000-0-02-1-7",
    "/kaggle/input/notebooks/thebeo182004/anomalyraddinov2-vindrcxr-1-1-0-1-8",
    "/kaggle/input/notebooks/thebeo182004/anomalyraddinov2-vindrcxr-5-1-0-1-9",
    "/kaggle/input/notebooks/chihoangf/anomalyraddinov2-vindrcxr-10-1-0-1-10",
    "/kaggle/input/notebooks/thebeo182004/anomalyraddinov2-vindrcxr-20-1-0-1-11"
]

cases = [
    '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11'
]

memory_bank_path = 'raddino_memory_bank.pt'

save_dir = '/kaggle/working/inference'
os.makedirs(save_dir, exist_ok=True)

import numpy as np

def save_inference_results(save_path: str, true_labels: np.ndarray, predicted_scores_per_k: dict):

    data_to_save = {
        'true_labels': true_labels
    }

    for k, strats in predicted_scores_per_k.items():
        for strategy_name, scores in strats.items():
            key = f"k_{k}_{strategy_name}"
            data_to_save[key] = scores 
        
    np.savez(save_path, **data_to_save)
    print(f"Successfully save at: {save_path}")

for case, mem_bank in zip(cases, memory_bank_dirs):
    
    # save_dir = os.path.join('/kaggle/working/inference', case)
    
    memory_bank_path = os.path.join(mem_bank, 'raddino_memory_bank.pt')

    true_labels, anomaly_scores = binary_classification_inference_aamb(
        raddino_checkpoint_path=raddino_checkpoint_path,
        memory_bank_path=memory_bank_path,
        normal_paths=test_normal_imgs,
        abnormal_paths=test_abnormal_imgs,
        batch_size=8,
        num_neighbours=[1, 2, 4, 8, 10, 15, 20]
    )

    save_inference_results(
        os.path.join(save_dir, f"inference_case_{case}.npz"),
        true_labels,
        anomaly_scores
    )

In [ ]:
!python /kaggle/working/run_inference.py